# 国会議事録 API ハンズオン（Section 06）

このノートブックでは、国会議事録 API からデータを取得し、
Python で確認・加工していく流れを体験します。

---

## ここでやること
- API からデータを取得する
- 取得パラメータを変えて結果の違いを見る
- 表形式に整形して中身を確認する
- CSV に保存して、他のツールでも使える形にする

> コードは用意されています。  
> **基本は「実行するだけ」でOK**。安心して進めてください。

In [ ]:
import requests

# 国会議事録 API のエンドポイント
BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

# まずはシンプルな条件で少しだけデータを取ってみる
params = {
    "recordPacking": "json",   # JSON形式で受け取る指定
    "any": "教育",             # キーワード（とりあえず「教育」で検索）
    "from": "2020-01-01",      # 検索開始日
    "until": "2025-12-31",     # 検索終了日
    "maximumRecords": 10       # 取得件数（まずは10件だけ）
}

# APIにリクエストを送信
response = requests.get(BASE_URL, params=params)

# エラーがあればここで例外を出す（デバッグしやすくするため）
response.raise_for_status()

# JSONデータとして読み込む
data = response.json()

# 国会議事録APIは、"speechRecord" というキーに発言データが入っている想定
records = data.get("speechRecord", [])

# 先頭数件だけ表示
for i, rec in enumerate(records[:5], start=1):
    print("--------------------------------")
    print(f"[{i}] 日付     : {rec.get('date')}")
    print(f"    委員会   : {rec.get('nameOfMeeting')}")
    print(f"    発言者   : {rec.get('speaker')}")
    text = rec.get("speech", "")
    print(f"    発言冒頭 : {text[:60]}{'...' if len(text) > 60 else ''}")


In [ ]:
# =========================================
# 初期セットアップ
# （外部ファイルを自動生成します）
# =========================================

import os

if not os.path.exists("output_layouts.py"):
    from textwrap import dedent

    with open("output_layouts.py", "w", encoding="utf-8") as f:
        f.write(dedent("""
        def show_card_layout(records):
            print("=== カード形式の出力例 ===")
            for i, rec in enumerate(records[:3], start=1):
                print("-" * 40)
                print(f"[{i}件目]")
                print(f"発言ID      : {rec.get('speechID')}")
                print(f"日付        : {rec.get('date')}")
                print(f"会議名      : {rec.get('nameOfMeeting')}")
                print(f"発言者      : {rec.get('speaker')}")
                speech = (rec.get("speech") or "")
                print("\\n【発言内容（冒頭）】")
                print(speech[:60] + ("..." if len(speech) > 60 else ""))


        def show_summary_layout(records):
            print("=== 1行サマリー形式の出力例 ===")
            for i, rec in enumerate(records[:3], start=1):
                speech = (rec.get("speech") or "")
                head = speech[:40] + ("..." if len(speech) > 40 else "")
                print(f"[{i}] {rec.get('date')} / {rec.get('nameOfMeeting')} / {rec.get('speaker')}")
                print(f"    → {head}")


        def show_table_layout(records):
            print("=== 表形式の出力例 ===")
            print("No | 日付        | 委員会名         | 発言者     | 発言冒頭")
            print("-" * 80)
            for i, rec in enumerate(records[:3], start=1):
                speech = (rec.get("speech") or "")
                head = speech[:20] + ("..." if len(speech) > 20 else "")
                print(f"{i:<2} | {rec.get('date',''):<10} | {rec.get('nameOfMeeting','')[:12]:<12} | "
                      f"{rec.get('speaker','')[:8]:<8} | {head}")
        """))

    print("output_layouts.py を生成しました")
else:
    print("output_layouts.py はすでに存在しています。")


In [ ]:
import requests

# ------------------------------------
# 🔽 ここだけ自由に変更して試してOK 🔽
# ------------------------------------
params = {
    "recordPacking": "json",   # JSON形式で受け取る（固定でOK）
    "any": "教育",             # 🔍 キーワード（例：「教育」「AI」「災害」など）
    "from": "2022-01-01",      # 📅 検索開始日
    "until": "2025-12-31",     # 検索終了日
    "maximumRecords": 20       # 📌 取得件数（増やしすぎ注意！ 20〜50がおすすめ）
}
# ------------------------------------
# ここより下は触らなくてOK
# ------------------------------------

BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

response = requests.get(BASE_URL, params=params)
response.raise_for_status()

data = response.json()
records = data.get("speechRecord", [])

print("================================")
print("検索条件")
print(f"キーワード     : {params['any']}")
print(f"期間           : {params['from']} ～ {params['until']}")
print(f"取得件数       : {params['maximumRecords']}")
print("================================")
print(f"実際に取得できた件数: {len(records)} 件")
print("")

# 先頭数件だけ表示
for i, rec in enumerate(records[:5], start=1):
    print("--------------------------------")
    print(f"[{i}] 日付     : {rec.get('date')}")
    print(f"    委員会   : {rec.get('nameOfMeeting')}")
    print(f"    発言者   : {rec.get('speaker')}")
    text = rec.get("speech", "")
    print(f"    発言冒頭 : {text[:60]}{'...' if len(text) > 60 else ''}")


In [ ]:
import requests
import pandas as pd
from google.colab import files

# ==============================
# API からデータ取得
# ==============================
BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

params = {
    "recordPacking": "json",
    "any": "AI",               # 🔍 キーワード（必要なら変更OK）
    "from": "2022-01-01",      # 📅 期間開始
    "until": "2025-12-31",     # 検索終了日
    "maximumRecords": 50       # 取得件数
}

response = requests.get(BASE_URL, params=params)
response.raise_for_status()

data = response.json()
records = data.get("speechRecord", [])

print(f"取得データ件数 : {len(records)} 件")

# ==============================
# DataFrame に整形
# ==============================
if len(records) == 0:
    print("データが取得できませんでした。パラメータを見直してください。")
else:
    def short_text(text, max_len=80):
        if text is None:
            return ""
        return text[:max_len] + "…" if len(text) > max_len else text

    df = pd.DataFrame([
        {
            "発言ID": r.get("speechID"),
            "日付": r.get("date"),
            "会議名": r.get("nameOfMeeting"),
            "発言者": r.get("speaker"),
            "党派": r.get("speakerGroup"),
            "発言冒頭": short_text(r.get("speech", ""))
        }
        for r in records
    ])

    print("DataFrame の先頭10行を表示します👇")
    display(df.head(10))


    # ==============================
    # CSV 保存 & ダウンロード
    # ==============================
    csv_filename = "kokkai_speech.csv"
    df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

    print(f"CSV に保存しました：{csv_filename}")
    print("ダウンロードを開始します…")
    files.download(csv_filename)


In [ ]:
import requests

# ------------------------------------
# 🔽 ここだけ自由に変更して試してOK 🔽
# ------------------------------------
params = {
    "recordPacking": "json",             # JSON形式で受け取る（固定でOK）
    "nameOfHouse": "衆議院",             # 衆議院 / 参議院 などで絞り込み
    "nameOfMeeting": "文部科学委員会",   # 委員会名で絞り込み（例：文部科学委員会）
    "any": "数学",                       # 🔍 キーワード（例：「教育」「AI」「災害」など）
    "from": "2022-01-01",                # 📅 検索開始日
    "until": "2025-12-31",               # 検索終了日
    "maximumRecords": 20                 # 📌 取得件数（増やしすぎ注意！ 20〜50がおすすめ）
}
# ------------------------------------
# ここより下は触らなくてOK
# ------------------------------------

BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

response = requests.get(BASE_URL, params=params)
response.raise_for_status()

data = response.json()
records = data.get("speechRecord", [])

print("================================")
print("検索条件")
print(f"キーワード     : {params['any']}")
print(f"期間           : {params['from']} ～ {params['until']}")
print(f"取得件数       : {params['maximumRecords']}")
print("================================")
print(f"実際に取得できた件数: {len(records)} 件")
print("")

for i, rec in enumerate(records, start=1):
    text = rec.get("speech", "")
    head = text[:300].replace('\n','')
    pos = head.find("数学")
    #print("LEN(head)=", len(head), "HIT=", "数学" in head, str(pos))
    if "数学" in head:
        print("--------------------------------")
        print(f"[{i}] 日付     : {rec.get('date')}")
        print(f"    委員会   : {rec.get('nameOfMeeting')}")
        print(f"    発言者   : {rec.get('speaker')}")
        print("    発言冒頭 :")
        for i in range(0, len(head), 50):
            print(head[i:i+50])

In [ ]:
import requests

# ------------------------------------
# 　100件以上のデータを取得する例（300件）
# ------------------------------------
all_records = []

for start in range(1, 1001, 10):
    params = {
        "recordPacking": "json",
        "any": "数学",
        "from": "2022-01-01",
        "until": "2025-12-31",
        "maximumRecords": 10,
        "startRecord": start
    }

    BASE_URL = "https://kokkai.ndl.go.jp/api/speech"

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        # これ以上データがないと判断して終了
        print(f"取得終了（startRecord={start}）")
        break

    data = response.json()
    records = data.get("speechRecord", [])
    all_records.extend(records)

print(len(all_records))

if len(all_records) == 0:
    print("データが取得できませんでした。パラメータを見直してください。")
else:
    df = pd.DataFrame([
       {
            "発言ID": r.get("speechID"),
            "日付": r.get("date"),
            "会議名": r.get("nameOfMeeting"),
            "発言者": r.get("speaker"),
            "党派": r.get("speakerGroup"),
            "発言": r.get("speech", "")
        }
        for r in all_records
    ])

    print("DataFrame の先頭10行を表示します👇")
    display(df.head(10))

    # ==============================
    # CSV 保存 & ダウンロード
    # ==============================
    csv_filename = "kokkai_speech_for_qa.csv"
    df.to_csv(csv_filename, index=False, encoding="utf-8-sig")

    print(f"CSV に保存しました：{csv_filename}")
    print("ダウンロードを開始します…")
    files.download(csv_filename)